# 🧉 Voz argentina para Mate — versión Kaggle (sigue desde donde quedó en Colab)

**Antes de correr**
1. Panel derecho → *Session options* → **Accelerator: GPU T4 x2** (o P100) y **Internet: On**.
2. *Add Input* → tu dataset **mate-voz** (con `dataset_mate.zip` y `last.ckpt`).
3. Arriba a la derecha: **Save Version → Save & Run All (Commit)**. Corre solo en segundo plano (hasta 12 h): podés cerrar la compu.
4. Cuando termine: en la versión guardada, pestaña **Output** → bajá `es_AR-mate-medium.onnx` y `es_AR-mate-medium.onnx.json`.

In [ ]:
# 1. Encontrar los archivos subidos (Kaggle los pone en /kaggle/input/... y DESCOMPRIME los .zip solo)
import glob, os
print('Lo que hay en /kaggle/input:')
!find /kaggle/input -maxdepth 4 | head -30
meta = glob.glob('/kaggle/input/**/metadata.csv', recursive=True)
zipf = glob.glob('/kaggle/input/**/dataset_mate.zip', recursive=True)
ckpt = glob.glob('/kaggle/input/**/*.ckpt', recursive=True)
assert meta or zipf, 'No encuentro los datos de la voz: agregá tu dataset con Add Input (panel derecho)'
assert ckpt, 'No encuentro last.ckpt: subilo al dataset y agregalo con Add Input'
CKPT = sorted(ckpt, key=lambda c: ('last' not in c, c))[0]
OUT = '/kaggle/working'
print('checkpoint:', CKPT)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Instalar Piper con entrenamiento (docs/TRAINING.md de OHF-Voice/piper1-gpl)
!mkdir -p /kaggle/temp
%cd /kaggle/temp
!apt-get -qq install -y build-essential cmake ninja-build > /dev/null 2>&1; true
!rm -rf /kaggle/temp/piper1-gpl && git clone -q https://github.com/OHF-Voice/piper1-gpl.git /kaggle/temp/piper1-gpl
%cd /kaggle/temp/piper1-gpl
!python3 -m pip -q install -e '.[train]'
!bash ./build_monotonic_align.sh
# espeakbridge (convierte texto a sonidos): ya compilado, del paquete oficial
!python3 -m pip -q download piper-tts --no-deps -d /kaggle/temp/whl
!cd /kaggle/temp/whl && rm -rf pw && unzip -q -o piper_tts-*.whl -d pw
!cp /kaggle/temp/whl/pw/piper/espeakbridge* /kaggle/temp/piper1-gpl/src/piper/
!cp -r /kaggle/temp/whl/pw/piper/espeak-ng-data /kaggle/temp/piper1-gpl/src/piper/ 2>/dev/null; true
!python3 -c "from piper import espeakbridge; import piper.train; print('piper train OK')"

In [ ]:
# 3. Datos de la voz 05 (143 frases): uso la carpeta ya descomprimida por Kaggle, o descomprimo el zip
import shutil
shutil.rmtree('/kaggle/temp/data', ignore_errors=True); os.makedirs('/kaggle/temp/data')
if meta:
    src = os.path.dirname(meta[0])
    shutil.copy(meta[0], '/kaggle/temp/data/metadata.csv')
    shutil.copytree(os.path.join(src, 'wav'), '/kaggle/temp/data/wav')
else:
    !unzip -q -o "{zipf[0]}" -d /kaggle/temp/data
!echo "Frases: $(wc -l < /kaggle/temp/data/metadata.csv) | audios: $(ls /kaggle/temp/data/wav | wc -l)"

In [ ]:
# (usa UNA sola GPU: Kaggle da 2, pero Piper entrena con una sin problemas)
# 4. Entrenar desde la época donde quedó en Colab (last.ckpt) hasta 2000.
#    max_time corta a las 11 h para que Kaggle llegue a guardar todo (su límite es 12 h).
!python3 -m piper.train fit \
  --data.voice_name es_AR-mate \
  --data.csv_path /kaggle/temp/data/metadata.csv \
  --data.audio_dir /kaggle/temp/data/wav/ \
  --model.sample_rate 22050 \
  --data.espeak_voice es-419 \
  --data.cache_dir /kaggle/temp/cache/ \
  --data.config_path {OUT}/es_AR-mate-medium.onnx.json \
  --data.batch_size 16 \
  --trainer.max_epochs 2000 \
  --trainer.max_time 00:11:00:00 \
  --trainer.devices 1 \
  --trainer.default_root_dir /kaggle/temp/checkpoints \
  --ckpt_path "{CKPT}"

In [ ]:
# 5. Exportar la voz para Mate y guardar el último checkpoint (por si querés seguir puliendo después)
last = sorted(glob.glob('/kaggle/temp/checkpoints/**/*.ckpt', recursive=True), key=os.path.getmtime)[-1]
print('Exporto:', last)
!python3 -m piper.train.export_onnx --checkpoint "{last}" --output-file {OUT}/es_AR-mate-medium.onnx
!cp "{last}" {OUT}/last.ckpt
!ls -la {OUT}

In [ ]:
# 6. Prueba de audio (queda en Output como prueba.wav)
%cd /kaggle/working
!python3 -m pip uninstall -y -q piper-tts && python3 -m pip install -q piper-tts
!cd /kaggle/working && python3 -c "import wave; from piper import PiperVoice; v = PiperVoice.load('/kaggle/working/es_AR-mate-medium.onnx'); w = wave.open('/kaggle/working/prueba.wav', 'wb'); v.synthesize_wav('¡Buenas! Soy Mate. Ya te anoté llamar al contador para el jueves. ¿Querés que te avise un rato antes?', w); w.close(); print('listo')"
from IPython.display import Audio
Audio('/kaggle/working/prueba.wav')

---
Datos: Google, OpenSLR 61 (CC BY-SA 4.0) — Guevara-Rukoz et al., LREC 2020. Base: voz Aldo de Piper.